# AEIC on ImageNet-1k — Ultra-Low Bitrate Perceptual Image Compression with Shallow Encoder

Runs the official code of **"Ultra-Low Bitrate Perceptual Image Compression with Shallow Encoder"**
(Zhang, Liu, Chen — CVPR 2026, [arXiv:2512.12229](https://arxiv.org/pdf/2512.12229v2),
[github.com/LuizScarlet/AEIC](https://github.com/LuizScarlet/AEIC)) on a subset of the
**ImageNet-1k validation set** and evaluates it with the same metrics as the paper:
**bpp, PSNR, MS-SSIM, LPIPS, DISTS, FID, KID**.

**Before running:** `Runtime -> Change runtime type -> T4 GPU` (any GPU runtime works).

Pipeline:
1. Clone the AEIC repo and install pinned dependencies.
2. Apply five small, documented code modifications (needed for Colab / ImageNet).
3. Download pretrained models (SD-Turbo, pruned VAE decoder, AEIC checkpoints).
4. Export `N_IMAGES` ImageNet-1k validation images as PNG.
5. Compress + reconstruct with AEIC-SE at 5 bitrate points.
6. Evaluate with the authors' `evaluate.py` (same script as in the paper, from MS-ILLM).
7. Build the results table, rate-distortion curves and qualitative figures.
8. *(Optional)* build the C++ rANS entropy coder and verify real bitstream sizes.

Approximate runtime on a free T4: **~1.5-2 h** with `N_IMAGES = 500` and all 5 checkpoints.
To make it faster, lower `N_IMAGES` (keep > 50, otherwise FID/KID are skipped by the eval
script) or remove entries from `CHECKPOINTS`.


In [ ]:
!nvidia-smi

## 1. Clone the repository and install dependencies

The repo pins old library versions (`diffusers==0.25.1` etc.) because the model code imports
module paths that were removed in newer `diffusers` (`diffusers.models.transformer_2d`,
`diffusers.models.unet_2d_blocks`). We keep those pins but **do not** install the pinned
`torch`/`triton`/`xformers` — Colab's preinstalled PyTorch is used instead (the code runs fine
on it, attention falls back to PyTorch SDPA). We also drop the repo's `neuralcompression`
dependency: it no longer builds on current Colab (its `compressai` dependency fails to compile),
and `evaluate.py` only uses one small function from it, which we inline in step 2.
`pytorch_lightning` is kept: the pruned VAE decoder checkpoint (`halfDecoder.ckpt`) is a
Lightning checkpoint, and `torch.load` needs the module to unpickle it.
Ignore pip's dependency-resolver warnings.

If the final `assert` fails, do **Runtime -> Restart session** and rerun this cell — pip
succeeded but Python had already imported the old library versions.

In [ ]:
%cd /content
!git clone https://github.com/LuizScarlet/AEIC.git

In [ ]:
!pip install -q \
    "diffusers==0.25.1" "transformers==4.46.3" "huggingface_hub==0.25.0" \
    "peft==0.13.2" "accelerate==1.9.0" "einops" "scipy" "omegaconf" "timm" \
    "pyiqa" "torchmetrics==1.6.2" "torch-fidelity" "pytorch_lightning" "datasets" "gdown"

import torch, diffusers, transformers
print("torch", torch.__version__, "| diffusers", diffusers.__version__,
      "| transformers", transformers.__version__, "| cuda", torch.cuda.is_available())
assert diffusers.__version__ == "0.25.1", \
    "Pinned versions are not active. Runtime -> Restart session, then rerun this cell."


## 2. Small code modifications

Five minimal patches to the original code (idempotent — the cell can safely be re-run, e.g. by
"Run all"; already-applied patches are skipped):

1. **`compress.py`** — `net.codec.update(force=True)` builds the quantized CDF tables and
   instantiates the C++ rANS entropy coder, which would require compiling the `src/cpp` engine.
   It is *only* needed for `--use_practical_entropy_coding`; without that flag AEIC reports the
   bpp estimated from the entropy model's likelihoods (the difference is verified in step 8).
   We guard the call behind the flag.
2. **`testing_utils.py`** — default `enable_xformers_memory_efficient_attention` to `False`
   (xformers is not installed; diffusers uses PyTorch scaled-dot-product attention instead).
3. **`testing_utils.py`** — default `compile_model` to `False`. ImageNet validation images have
   *varying resolutions*, so `torch.compile` would recompile for nearly every image and make the
   run dramatically slower instead of faster.
4. **`evaluate.py`** — its only use of the `neuralcompression` package (which fails to build on
   current Colab) is `update_patch_fid`, the patch-wise FID/KID protocol ("FID/256", Mentzer et
   al. 2020, as used by MS-ILLM and this paper). We inline a faithful port: non-overlapping
   256 px patches, plus a second half-patch-offset pass for images larger than 1.5x the patch
   size, converted to uint8 before updating the torchmetrics FID/KID objects.
5. **`evaluate.py`** — `KernelInceptionDistance(subset_size=50)` instead of the default 1000.
   ImageNet images are small (typically 1-2 patches of 256 px each), so the default subset size
   can exceed the total number of patches and make `.compute()` fail. The subset size affects
   only the variance of the KID estimate, not its expectation.

In [ ]:
from pathlib import Path

def patch(path, old, new):
    p = Path(path); src = p.read_text()
    if new in src:
        print("already patched:", path)
        return
    assert old in src, f"pattern not found in {path}"
    p.write_text(src.replace(old, new, 1))
    print("patched:", path)

patch("/content/AEIC/src/compress.py",
      "    net.codec.update(force=True)",
      "    if args.use_practical_entropy_coding:\n        net.codec.update(force=True)")

patch("/content/AEIC/src/my_utils/testing_utils.py",
      '"--enable_xformers_memory_efficient_attention", default=True',
      '"--enable_xformers_memory_efficient_attention", default=False')

patch("/content/AEIC/src/my_utils/testing_utils.py",
      '"--compile_model", default=True',
      '"--compile_model", default=False')

# Local port of neuralcompression.metrics.update_patch_fid (FID/256, Mentzer et al. 2020):
# the package no longer builds on Colab and evaluate.py needs only this one function.
Path("/content/AEIC/src/patch_fid_local.py").write_text('''
import torch
import torch.nn.functional as F


def _to_uint8_patches(images, patch_size):
    patches = (F.unfold(images, kernel_size=patch_size, stride=patch_size)
               .permute(0, 2, 1).reshape(-1, 3, patch_size, patch_size))
    return (patches * 255.0).round().clamp(0, 255).to(torch.uint8)


def update_patch_fid(input_images, pred, fid_metric=None, kid_metric=None, patch_size=256):
    # Port of neuralcompression.metrics.update_patch_fid (FID/256 protocol)
    patch_count = 0
    for offset in (0, patch_size // 2):
        if offset and not (input_images.shape[2] >= 1.5 * patch_size
                           and input_images.shape[3] >= 1.5 * patch_size):
            break
        real = _to_uint8_patches(input_images[:, :, offset:, offset:], patch_size)
        fake = _to_uint8_patches(pred[:, :, offset:, offset:], patch_size)
        patch_count += real.shape[0]
        if fid_metric is not None:
            fid_metric.update(real, real=True)
            fid_metric.update(fake, real=False)
        if kid_metric is not None:
            kid_metric.update(real, real=True)
            kid_metric.update(fake, real=False)
    return patch_count
''')
print("wrote: /content/AEIC/src/patch_fid_local.py")

patch("/content/AEIC/src/evaluate.py",
      "from neuralcompression.metrics import update_patch_fid",
      "from patch_fid_local import update_patch_fid")

patch("/content/AEIC/src/evaluate.py",
      "kid_metric = KernelInceptionDistance().to(device)",
      "kid_metric = KernelInceptionDistance(subset_size=50).to(device)")

## 3. Download pretrained models

* **SD-Turbo** — AEIC only loads the `vae` and `unet` sub-modules (no text encoder), so we
  download just those (~4 GB).
* **Pruned ("half") VAE decoder** from AdcSR (used as the lightweight final decoder).
* **AEIC checkpoints** from the authors' Google Drive folder — `AEIC_{ME,SE}_ft{2,4,8,16,32}.pkl`,
  i.e. 5 bitrate points per encoder variant. If `gdown` hits a Drive quota error, download the
  folder manually in a browser and upload the `.pkl` files to `/content/AEIC_ckpts/`.

In [ ]:
from huggingface_hub import snapshot_download, hf_hub_download

sd_path = snapshot_download(repo_id="stabilityai/sd-turbo",
                            allow_patterns=["model_index.json", "vae/*", "unet/*"])
vae_dec_path = hf_hub_download(repo_id="Guaishou74851/AdcSR",
                               filename="weight/pretrained/halfDecoder.ckpt")
print("SD-Turbo:", sd_path)
print("half VAE decoder:", vae_dec_path)

In [ ]:
!gdown --folder "https://drive.google.com/drive/folders/1vioCW4EIxQiuLkWHKj7xbMi7WAVcWqJI" -O /content/AEIC_ckpts
!ls -lh /content/AEIC_ckpts

## 4. Prepare the ImageNet-1k validation subset

The official [`ILSVRC/imagenet-1k`](https://huggingface.co/datasets/ILSVRC/imagenet-1k) dataset
on Hugging Face is **gated**: log in to HF, accept the terms on the dataset page, and create a
read token (https://huggingface.co/settings/tokens). Paste it below — or just press Enter to use
the ungated mirror `mrm8488/ImageNet1K-val` (same 50k validation images).

We **stream** the dataset (no 7 GB download) and export the first `N_IMAGES` images whose short
side is at least `MIN_SIDE` px as lossless PNG (the repo's `compress.py` globs `*.png`). The size
filter mirrors the paper's full-resolution protocol and is required by MS-SSIM, which needs
inputs larger than ~160 px after its 4 dyadic downsamplings.

In [ ]:
N_IMAGES = 500        # images to evaluate on (keep > 50 so FID/KID are computed)
MIN_SIDE = 256        # skip images with a shorter side than this
DATA_DIR = "/content/data/imagenet_val"
OUT_ROOT = "/content/outputs"
LOG_DIR  = "/content/logs"

import os
from getpass import getpass
HF_TOKEN = os.environ.get("HF_TOKEN") or getpass("HF token (Enter to skip -> ungated mirror): ") or None

In [ ]:
import os
from datasets import load_dataset

os.makedirs(DATA_DIR, exist_ok=True)

def open_stream():
    if HF_TOKEN:
        try:
            return load_dataset("ILSVRC/imagenet-1k", split="validation",
                                streaming=True, token=HF_TOKEN)
        except Exception as e:
            print("Official imagenet-1k not accessible:", e)
    print("Using ungated mirror mrm8488/ImageNet1K-val")
    return load_dataset("mrm8488/ImageNet1K-val", split="train", streaming=True)

stream = open_stream()
saved = skipped = 0
for sample in stream:
    img = sample["image"]
    if min(img.size) < MIN_SIDE:
        skipped += 1
        continue
    img.convert("RGB").save(os.path.join(DATA_DIR, f"img_{saved:05d}.png"))
    saved += 1
    if saved % 100 == 0:
        print(f"saved {saved}/{N_IMAGES}")
    if saved >= N_IMAGES:
        break
print(f"Done: {saved} PNGs in {DATA_DIR} ({skipped} skipped, shorter side < {MIN_SIDE}px)")

## 5. Run AEIC on the subset

We run the authors' `compress.py` once per checkpoint (5 bitrate points of the shallow-encoder
variant **AEIC-SE** by default — switch `CODEC_TYPE` to `"AEIC-ME"` for the moderate encoder).
Each run reconstructs every image and reports the **estimated bpp** from the entropy model.
`compress.py` must be executed with `src/` as the working directory (the repo's imports assume
it). Logs are kept in `LOG_DIR`.

In [ ]:
CODEC_TYPE  = "AEIC-SE"                          # "AEIC-SE" (shallow) or "AEIC-ME" (moderate)
CHECKPOINTS = ["ft2", "ft4", "ft8", "ft16", "ft32"]  # 5 rate points, highest -> lowest bpp

In [ ]:
import os, re, subprocess

os.makedirs(LOG_DIR, exist_ok=True)
avg_bpp = {}

for tag in CHECKPOINTS:
    name    = f"{CODEC_TYPE.replace('-', '_')}_{tag}"
    ckpt    = f"/content/AEIC_ckpts/{name}.pkl"
    rec_dir = f"{OUT_ROOT}/{name}/rec"
    bin_dir = f"{OUT_ROOT}/{name}/bin"
    assert os.path.exists(ckpt), f"checkpoint not found: {ckpt}"

    cmd = ["python", "compress.py",
           f"--sd_path={sd_path}", f"--img_path={DATA_DIR}",
           f"--rec_path={rec_dir}", f"--bin_path={bin_dir}",
           f"--codec_type={CODEC_TYPE}", f"--codec_path={ckpt}",
           f"--vae_decoder_path={vae_dec_path}"]

    print(f"\n=== {name} ===")
    log_path = f"{LOG_DIR}/compress_{name}.log"
    n_done = 0
    with open(log_path, "w") as log:
        proc = subprocess.Popen(cmd, cwd="/content/AEIC/src",
                                stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in proc.stdout:
            log.write(line)
            if line.startswith("[Processing]"):
                n_done += 1
                if n_done % 50 == 0:
                    print(f"  {n_done} images done")
            elif "out of memory" in line or "Error" in line:
                print(" ", line.strip())
        ret = proc.wait()
    assert ret == 0, f"compress.py failed for {name} - inspect {log_path}"

    tail = open(log_path).read().split("AVG Results")[-1]
    avg_bpp[name] = float(re.search(r"BPP: ([0-9.eE+-]+)", tail).group(1))
    print(f"  average estimated bpp: {avg_bpp[name]:.5f}")

avg_bpp

## 6. Evaluate with the paper's metrics

`src/evaluate.py` is the authors' evaluation script (adapted from MS-ILLM). Per image pair it
computes **PSNR** and **MS-SSIM** (pixel fidelity), **LPIPS** (AlexNet) and **DISTS** (deep
perceptual similarity), and accumulates patch-wise **FID** and **KID** over the whole set
(realism / distribution match — only reported when there are more than 50 images). The first
run downloads the metric networks' weights.

In [ ]:
import json, re, subprocess

results = {}
for tag in CHECKPOINTS:
    name    = f"{CODEC_TYPE.replace('-', '_')}_{tag}"
    rec_dir = f"{OUT_ROOT}/{name}/rec"
    print(f"\n=== evaluating {name} ===")
    out = subprocess.run(["python", "evaluate.py",
                          f"--gt_dir={DATA_DIR}", f"--recon_dir={rec_dir}"],
                         cwd="/content/AEIC/src", capture_output=True, text=True)
    if out.returncode != 0:
        print(out.stdout[-2000:]); print(out.stderr[-2000:])
        raise RuntimeError(f"evaluate.py failed for {name}")
    metrics = dict(re.findall(r"^([a-z_]+): (-?[0-9.eE+-]+)$", out.stdout, re.M))
    results[name] = {"bpp": avg_bpp[name], **{k: float(v) for k, v in metrics.items()}}
    print(results[name])

with open("/content/results_imagenet.json", "w") as f:
    json.dump(results, f, indent=2)

## 7. Results: table, rate-distortion curves, qualitative examples

In [ ]:
import pandas as pd

cols = ["bpp", "psnr", "ms_ssim", "lpips", "dists", "fid", "kid_mean", "kid_std"]
df = pd.DataFrame(results).T.reindex(columns=cols)
df.to_csv("/content/results_imagenet.csv")
df.round(5)

In [ ]:
import os
import matplotlib.pyplot as plt

# Authors' AEIC-SE benchmark numbers (results/AEIC_SE.txt in the repo) for reference
REF_KODAK = {
    "bpp":     [0.0040981, 0.0079119, 0.0143318, 0.0240701, 0.0370180],
    "psnr":    [18.99339, 20.17073, 21.30055, 22.15809, 22.89515],
    "ms_ssim": [0.58295, 0.66216, 0.72879, 0.78113, 0.82160],
    "lpips":   [0.34155, 0.27327, 0.21539, 0.17500, 0.14680],
    "dists":   [0.15487, 0.12743, 0.10768, 0.09150, 0.08149],
}
REF_CLIC = {
    "bpp":      [0.0032721, 0.0059195, 0.0103894, 0.0177338, 0.0282134],
    "psnr":     [21.16354, 22.58472, 23.91150, 25.08380, 25.98935],
    "ms_ssim":  [0.70447, 0.76217, 0.81078, 0.85124, 0.88037],
    "lpips":    [0.27180, 0.21537, 0.17219, 0.13820, 0.11501],
    "dists":    [0.11392, 0.09455, 0.07850, 0.06473, 0.05581],
    "fid":      [7.35, 5.29, 4.00, 2.95, 2.45],
    "kid_mean": [0.0012510, 0.0007672, 0.0005089, 0.0002478, 0.0001725],
}

n_imgs = len(os.listdir(DATA_DIR))
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
for ax, m in zip(axes.flat, ["psnr", "ms_ssim", "lpips", "dists", "fid", "kid_mean"]):
    if m in df.columns and df[m].notna().any():
        ax.plot(df["bpp"], df[m], "o-", lw=2, label=f"ImageNet-1k val, {n_imgs} imgs (ours)")
    for ref, lbl, style in [(REF_KODAK, "Kodak (paper)", "s--"), (REF_CLIC, "CLIC2020 (paper)", "^--")]:
        if m in ref:
            ax.plot(ref["bpp"], ref[m], style, alpha=0.6, label=lbl)
    ax.set_xlabel("bpp"); ax.set_ylabel(m); ax.grid(alpha=0.3); ax.legend(fontsize=8)
fig.suptitle(f"{CODEC_TYPE}: rate-distortion-perception on ImageNet-1k vs. paper benchmarks")
fig.tight_layout()
fig.savefig("/content/rd_curves_imagenet.png", dpi=200)
plt.show()

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

n_show = 3
low  = f"{CODEC_TYPE.replace('-', '_')}_{CHECKPOINTS[0]}"
high = f"{CODEC_TYPE.replace('-', '_')}_{CHECKPOINTS[-1]}"
names = sorted(os.listdir(DATA_DIR))[:n_show]

fig, axes = plt.subplots(n_show, 3, figsize=(12, 4 * n_show))
for r, fname in enumerate(names):
    panels = [(DATA_DIR, "original (PNG)"),
              (f"{OUT_ROOT}/{low}/rec",  f"{low}  ({df.loc[low, 'bpp']:.4f} bpp)"),
              (f"{OUT_ROOT}/{high}/rec", f"{high} ({df.loc[high, 'bpp']:.4f} bpp)")]
    for c, (folder, title) in enumerate(panels):
        axes[r, c].imshow(Image.open(os.path.join(folder, fname)))
        axes[r, c].set_title(title, fontsize=9)
        axes[r, c].axis("off")
fig.tight_layout()
fig.savefig("/content/qualitative_imagenet.png", dpi=200)
plt.show()

In [ ]:
!zip -j -q /content/aeic_imagenet_artifacts.zip \
    /content/results_imagenet.csv /content/results_imagenet.json \
    /content/rd_curves_imagenet.png /content/qualitative_imagenet.png
!zip -r -q /content/aeic_imagenet_artifacts.zip /content/logs
print("Download aeic_imagenet_artifacts.zip from the Files sidebar (left, folder icon).")

## 8. *(Optional)* Practical entropy coding — verify real bitstream sizes

Everything above reports bpp **estimated** from the entropy model. To verify that real
bitstreams match, build the repo's C++ rANS coder (cmake fetches pybind11 automatically) and
re-run one checkpoint on a small subset with `--use_practical_entropy_coding`, which writes
actual `.bin` files and measures their size. Expect the real bpp to be slightly *higher*
(arithmetic-coding overhead plus a small per-image header), especially on small images.

In [ ]:
!apt-get -qq install -y cmake g++ > /dev/null

from pathlib import Path

# The build fetches pybind11 v2.10.4, which predates Python 3.12 and fails to compile
# against it in the code paths py_rans.cpp uses -> bump to v2.13.6 (Python 3.8-3.13, NumPy 2).
cm_in = Path("/content/AEIC/src/cpp/3rdparty/pybind11/CMakeLists.txt.in")
cm_in.write_text(cm_in.read_text().replace("v2.10.4", "v2.13.6"))
print("pybind11 tag bumped to v2.13.6")

# The repo also builds with -Werror; newer g++ warnings would abort the build. Strip it.
cm = Path("/content/AEIC/src/cpp/CMakeLists.txt")
cm.write_text(cm.read_text().replace(" -Werror", "").replace("/WX", ""))
print("stripped -Werror")

!rm -rf /content/AEIC/src/build && mkdir -p /content/AEIC/src/build
!cd /content/AEIC/src/build && cmake ../cpp -DCMAKE_BUILD_TYPE=Release -Wno-dev 2>&1 | tail -3
!cd /content/AEIC/src/build && make -j2 2>&1 | tail -40
!ls -l /content/AEIC/src/codec/ | grep MLCodec

import glob
assert glob.glob("/content/AEIC/src/codec/MLCodec_rans*.so"), \
    "MLCodec_rans did not build - the make output above shows the compiler error"
print("entropy-coding engine ready")

In [ ]:
import os, re, shutil, subprocess

PRACTICAL_N   = 50
PRACTICAL_TAG = CHECKPOINTS[len(CHECKPOINTS) // 2]

small_dir = "/content/data/imagenet_val_small"
os.makedirs(small_dir, exist_ok=True)
for f in sorted(os.listdir(DATA_DIR))[:PRACTICAL_N]:
    shutil.copy(os.path.join(DATA_DIR, f), small_dir)

name = f"{CODEC_TYPE.replace('-', '_')}_{PRACTICAL_TAG}"
cmd = ["python", "compress.py",
       f"--sd_path={sd_path}", f"--img_path={small_dir}",
       f"--rec_path={OUT_ROOT}/{name}_practical/rec",
       f"--bin_path={OUT_ROOT}/{name}_practical/bin",
       f"--codec_type={CODEC_TYPE}",
       f"--codec_path=/content/AEIC_ckpts/{name}.pkl",
       f"--vae_decoder_path={vae_dec_path}",
       "--use_practical_entropy_coding"]
out = subprocess.run(cmd, cwd="/content/AEIC/src", capture_output=True, text=True)
if out.returncode != 0:
    print(out.stdout[-1500:])
    print(out.stderr[-1500:])
    raise RuntimeError("practical-coding run failed - see output above")
print(out.stdout[-1200:])
real_bpp = float(re.search(r"BPP: ([0-9.eE+-]+)", out.stdout.split("AVG Results")[-1]).group(1))
print(f"\nreal bitstream bpp ({PRACTICAL_N} imgs): {real_bpp:.5f}")
print(f"estimated bpp (full subset):        {avg_bpp[name]:.5f}")

## 9. Extras — browse compressions, zoom into details, JPEG baseline

Optional material for the presentation/defense. The reconstructions for all five rate points
already exist under `OUT_ROOT`, so these cells only read and arrange them:
* **contact sheets** — every image as original + all 5 rate points (bpp in the headers);
* **zoom viewer** — magnified crop across rates, where texture hallucination becomes visible;
* **JPEG baseline** — encode the same images with JPEG at quality 1 (its floor) and evaluate
  with the same script: classical coding cannot even reach ultra-low bitrates.

In [ ]:
import os, json, math
from PIL import Image
import matplotlib.pyplot as plt

N_GRID, PER_SHEET = 30, 6            # render first N_GRID subset images, PER_SHEET rows/sheet
COMP_DIR = "/content/comparisons"
os.makedirs(COMP_DIR, exist_ok=True)

res  = json.load(open("/content/results_imagenet.json"))
tags = list(CHECKPOINTS)                              # ft2 .. ft32 = high -> low bitrate
cols = ["original"] + tags
names = sorted(os.listdir(DATA_DIR))[:N_GRID]

def img_path(tag, fname):
    if tag == "original":
        return os.path.join(DATA_DIR, fname)
    return f"{OUT_ROOT}/{CODEC_TYPE.replace('-', '_')}_{tag}/rec/{fname}"

for s in range(math.ceil(len(names) / PER_SHEET)):
    chunk = names[s*PER_SHEET:(s+1)*PER_SHEET]
    fig, axes = plt.subplots(len(chunk), len(cols), figsize=(3.2*len(cols), 2.6*len(chunk)))
    if len(chunk) == 1:
        axes = axes[None, :]
    for r, fname in enumerate(chunk):
        for c, tag in enumerate(cols):
            axes[r, c].imshow(Image.open(img_path(tag, fname)))
            axes[r, c].axis("off")
            if r == 0:
                key = f"{CODEC_TYPE.replace('-', '_')}_{tag}"
                axes[r, c].set_title("original" if tag == "original"
                    else f"{tag} ({res[key]['bpp']:.4f} bpp)", fontsize=10)
    fig.tight_layout()
    fig.savefig(f"{COMP_DIR}/comparison_{s+1:02d}.png", dpi=130, bbox_inches="tight")
    plt.close(fig)
    print("saved sheet", s + 1)

!zip -j -q /content/comparisons.zip /content/comparisons/*.png
print("download /content/comparisons.zip from the Files sidebar")

In [ ]:
IDX  = 7                  # which image (0 .. N_IMAGES-1) - change and re-run to explore
CROP = 0.5, 0.4, 220      # relative center x, y + crop size in px

fname = sorted(os.listdir(DATA_DIR))[IDX]

def crop(im):
    cx, cy, s = int(CROP[0]*im.width), int(CROP[1]*im.height), CROP[2] // 2
    box = (max(cx-s, 0), max(cy-s, 0), min(cx+s, im.width), min(cy+s, im.height))
    return im.crop(box).resize((360, 360), Image.NEAREST)

fig, axes = plt.subplots(2, len(cols), figsize=(3.0*len(cols), 6.2))
for c, tag in enumerate(cols):
    im = Image.open(img_path(tag, fname))
    axes[0, c].imshow(im)
    axes[1, c].imshow(crop(im))
    key = f"{CODEC_TYPE.replace('-', '_')}_{tag}"
    axes[0, c].set_title("original" if tag == "original"
        else f"{tag} ({res[key]['bpp']:.4f} bpp)", fontsize=10)
    for ax in (axes[0, c], axes[1, c]):
        ax.axis("off")
fig.tight_layout()
fig.savefig("/content/detail_comparison.png", dpi=160, bbox_inches="tight")
plt.show()

In [ ]:
# JPEG baseline: can classical coding even operate at these bitrates?
import os, subprocess
from PIL import Image

JPEG_DIR = "/content/outputs/jpeg_q1"
os.makedirs(f"{JPEG_DIR}/rec", exist_ok=True)
bits = px = 0
for f in sorted(os.listdir(DATA_DIR)):
    im = Image.open(os.path.join(DATA_DIR, f))
    jp = os.path.join(JPEG_DIR, f.replace(".png", ".jpg"))
    im.save(jp, "JPEG", quality=1)               # JPEG's absolute floor
    bits += os.path.getsize(jp) * 8
    px   += im.width * im.height
    Image.open(jp).save(os.path.join(JPEG_DIR, "rec", f))   # PNG names for evaluate.py
print(f"JPEG q=1 average bpp: {bits/px:.4f}")

out = subprocess.run(["python", "evaluate.py", f"--gt_dir={DATA_DIR}",
                      f"--recon_dir={JPEG_DIR}/rec"],
                     cwd="/content/AEIC/src", capture_output=True, text=True)
print(out.stdout[-400:])

## Troubleshooting

* **CUDA out of memory** — add `--vae_decoder_tiled_size=128` to the `compress.py` command
  (flag already exists in `testing_utils.py`); ImageNet images are small, so this is unlikely.
* **gdown quota exceeded** — open the Drive folder in a browser, download the `.pkl` files you
  need and upload them to `/content/AEIC_ckpts/`.
* **Colab disconnects** — results so far are kept in `/content/logs` and `/content/outputs`;
  rerun only the missing checkpoints (edit `CHECKPOINTS`).
* **LoRA loading errors** — make sure `peft==0.13.2` got installed (cell 1) and restart the
  runtime so the pinned versions are the ones imported.

**For the report:** copy the table from `results_imagenet.csv`, the curves from
`rd_curves_imagenet.png` and examples from `qualitative_imagenet.png` into the IEEE document.
